# ARC/ATLAS SynthSR Upsampling

Applies **SynthSR** (Iglesias et al., *Science Advances* 2023) to the `test_lores` set —
198 naturally low-quality MRIs that were held out entirely from v3 training.

SynthSR produces a synthetic 1mm isotropic T1w from any input MRI quality/contrast,
without requiring paired training data.

**Pipeline:**
1. Input: `test_lores/t1/` — ANTs-registered, normalised [0,1] T1s
2. SynthSR enhances each image → synthetic high-quality T1
3. Re-normalise output to [0,1] (v3 format)
4. Copy masks unchanged from `test_lores/masks/` (same subject, same MNI space)
5. Output: `Upsampled_LowRes/t1/` and `Upsampled_LowRes/masks/`

**Research question:** Does SR preprocessing recover the segmentation accuracy lost
due to natural image degradation?

In [1]:
import sys, os, shutil, subprocess
from pathlib import Path
import numpy as np
import nibabel as nib

# --- Paths ---
V4_ROOT    = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4")
SPLIT_BASE = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data")
SRC_T1_DIR   = SPLIT_BASE / "test_lores/t1"
SRC_MASK_DIR = SPLIT_BASE / "test_lores/masks"
OUT_DIR      = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Upsampled_LowRes")
OUT_T1_DIR   = OUT_DIR / "t1"
OUT_MASK_DIR = OUT_DIR / "masks"
TMP_DIR      = OUT_DIR / "_synthsr_tmp"

for d in [OUT_T1_DIR, OUT_MASK_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Import normalize_t1 from v4 prep_utils
sys.path.insert(0, str(V4_ROOT / "src" / "data_prep"))
from prep_utils import normalize_t1

OVERWRITE = False
N_JOBS    = 1   # SynthSR runs its own internal parallelism; set >1 only if you have many GPUs

# Sanity checks
src_t1s = sorted(SRC_T1_DIR.glob("*_T1w_MNI_norm.nii.gz"))
assert len(src_t1s) > 0, f"No T1s found in {SRC_T1_DIR}"
print(f"Source T1s: {len(src_t1s)}")
print(f"Source masks: {len(list(SRC_MASK_DIR.glob('*.nii.gz')))}")

Source T1s: 198
Source masks: 198


In [2]:
# Locate mri_synthsr and configure GPU/CPU strategy
import subprocess, copy, os
from pathlib import Path

FREESURFER_HOME = Path(os.environ.get("FREESURFER_HOME",
                       str(Path.home() / "stroke_cleaned/freesurfer")))
fs_bin = FREESURFER_HOME / "bin"
if str(fs_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = str(fs_bin) + ":" + os.environ.get("PATH", "")

result = subprocess.run(["which", "mri_synthsr"], capture_output=True, text=True)
assert result.returncode == 0, f"mri_synthsr not found. Set FREESURFER_HOME and source SetUpFreeSurfer.sh"
synthsr_bin = Path(result.stdout.strip())
print(f"mri_synthsr: {synthsr_bin}")

# Detect GPUs via nvidia-smi
gpu_result = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
if gpu_result.returncode == 0 and gpu_result.stdout.strip():
    GPU_IDS = [line.split(",")[0].strip() for line in gpu_result.stdout.strip().splitlines()]
    print(f"GPUs available: {GPU_IDS}")
    USE_CPU = False
else:
    GPU_IDS = []
    USE_CPU = True
    print("No GPUs found — will use CPU")

N_PARALLEL = max(len(GPU_IDS), 1)   # one worker per GPU; 1 if CPU-only
N_CPU_THREADS = os.cpu_count()       # used only in CPU fallback
print(f"Parallel workers: {N_PARALLEL}  |  CPU threads: {N_CPU_THREADS}  |  CPU mode: {USE_CPU}")


mri_synthsr: /home/rbielski/stroke_cleaned/freesurfer/bin/mri_synthsr
GPUs available: ['0', '1']
Parallel workers: 2  |  CPU threads: 32  |  CPU mode: False


In [ ]:
# Run SynthSR — GPU parallel (one subject per GPU) or CPU threaded
import concurrent.futures, copy

# Base env with FreeSurfer vars
base_env = copy.copy(os.environ)
base_env["FREESURFER_HOME"] = str(FREESURFER_HOME)
base_env["PATH"]            = str(FREESURFER_HOME / "bin") + ":" + base_env.get("PATH", "")
base_env.setdefault("SUBJECTS_DIR", str(FREESURFER_HOME / "subjects"))
base_env.setdefault("MNI_DIR",      str(FREESURFER_HOME / "mni"))

def run_synthsr(args):
    t1_path, gpu_id = args
    key    = t1_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    out_t1 = OUT_T1_DIR / t1_path.name
    if not OVERWRITE and out_t1.exists():
        return key, True, "skipped"

    tmp_out = TMP_DIR / f"{key}_synthsr_raw.nii.gz"
    env = copy.copy(base_env)

    if USE_CPU:
        cmd = [str(synthsr_bin), "--i", str(t1_path), "--o", str(tmp_out),
               "--cpu", "--threads", str(N_CPU_THREADS)]
    else:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        cmd = [str(synthsr_bin), "--i", str(t1_path), "--o", str(tmp_out)]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
        output = (result.stdout + result.stderr).strip()
        if result.returncode != 0 or not tmp_out.exists():
            return key, False, output[-600:] if output else "(no output)"

        img  = nib.load(str(tmp_out))
        norm = normalize_t1(img.get_fdata(dtype=np.float32))
        nib.save(nib.Nifti1Image(norm, img.affine, img.header), str(out_t1))
        tmp_out.unlink(missing_ok=True)
        return key, True, "ok"

    except Exception as e:
        tmp_out.unlink(missing_ok=True)
        return key, False, str(e)

todo = [p for p in src_t1s if OVERWRITE or not (OUT_T1_DIR / p.name).exists()]
print(f"To process: {len(todo)}  (skipping {len(src_t1s) - len(todo)} already done)")
print(f"Running with {N_PARALLEL} parallel worker(s) on {'GPU(s) ' + str(GPU_IDS) if not USE_CPU else 'CPU'}\n")

# Assign GPU IDs round-robin; CPU jobs all get gpu_id=None
if USE_CPU:
    jobs = [(p, None) for p in todo]
else:
    jobs = [(p, GPU_IDS[i % len(GPU_IDS)]) for i, p in enumerate(todo)]

ok = fail = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
    futures = {pool.submit(run_synthsr, j): j[0] for j in jobs}
    for i, fut in enumerate(concurrent.futures.as_completed(futures)):
        key, success, msg = fut.result()
        status = "OK" if success and msg != "skipped" else ("SKIP" if msg == "skipped" else "FAIL")
        print(f"[{i+1}/{len(todo)}] {status}  {key}" + (f"\n  {msg}" if status == "FAIL" else ""), flush=True)
        if success:
            ok += 1
        else:
            fail += 1

print(f"\nDone — processed: {ok}  failed: {fail}  skipped: {len(src_t1s)-len(todo)}")


To process: 198  (skipping 0 already done)
Running with 2 parallel worker(s) on GPU(s) ['0', '1']

[1/198] OK  sub-M2044_ses-2150
[2/198] OK  sub-M2037_ses-275
[3/198] OK  sub-M2046_ses-2499
[4/198] OK  sub-M2048_ses-215
[5/198] OK  sub-M2079_ses-217
[6/198] OK  sub-M2116_ses-261
[7/198] OK  sub-M2134_ses-1026
[8/198] OK  sub-M2120_ses-749
[9/198] OK  sub-M2146_ses-2612
[10/198] OK  sub-M2153_ses-1566
[11/198] OK  sub-M2158_ses-404
[12/198] OK  sub-M2156_ses-2653
[13/198] OK  sub-M2160_ses-3078
[14/198] OK  sub-M2204_ses-577
[15/198] OK  sub-M2208_ses-446
[16/198] OK  sub-M2221_ses-2045
[17/198] OK  sub-M2228_ses-456
[18/198] OK  sub-M2246_ses-853
[19/198] OK  sub-M2252_ses-738
[20/198] OK  sub-M2281_ses-500
[21/198] OK  sub-r001s009_ses-1
[22/198] OK  sub-r001s034_ses-1
[23/198] OK  sub-r002s003_ses-1
[24/198] OK  sub-r002s002_ses-1
[25/198] OK  sub-r002s004_ses-1
[26/198] OK  sub-r002s007_ses-1
[27/198] OK  sub-r002s008_ses-1
[28/198] OK  sub-r002s009_ses-1
[29/198] OK  sub-r002s011_

In [ ]:
# Copy masks from test_lores/masks/ -> Upsampled_LowRes/masks/
# No processing needed: same subject, same MNI registration, SR doesn't move voxels.

src_masks = sorted(SRC_MASK_DIR.glob("*_lesion_mask_MNI_clean.nii.gz"))
print(f"Source masks: {len(src_masks)}")

copied = skipped = 0
for mask_path in src_masks:
    dst = OUT_MASK_DIR / mask_path.name
    # Only copy if the corresponding SR T1 was successfully produced
    key = mask_path.name.replace("_lesion_mask_MNI_clean.nii.gz", "")
    t1_done = (OUT_T1_DIR / f"{key}_T1w_MNI_norm.nii.gz").exists()
    if not t1_done:
        print(f"[skip mask] {key} — SR T1 not found")
        continue
    if not OVERWRITE and dst.exists():
        skipped += 1
        continue
    shutil.copy2(str(mask_path), str(dst))
    copied += 1

print(f"Masks — copied: {copied}, skipped: {skipped}")

Source masks: 198
[skip mask] sub-M2037_ses-275 — SR T1 not found
[skip mask] sub-M2044_ses-2150 — SR T1 not found
[skip mask] sub-M2046_ses-2499 — SR T1 not found
[skip mask] sub-M2048_ses-215 — SR T1 not found
[skip mask] sub-M2079_ses-217 — SR T1 not found
[skip mask] sub-M2116_ses-261 — SR T1 not found
[skip mask] sub-M2120_ses-749 — SR T1 not found
[skip mask] sub-M2134_ses-1026 — SR T1 not found
[skip mask] sub-M2146_ses-2612 — SR T1 not found
[skip mask] sub-M2153_ses-1566 — SR T1 not found
[skip mask] sub-M2156_ses-2653 — SR T1 not found
[skip mask] sub-M2158_ses-404 — SR T1 not found
[skip mask] sub-M2160_ses-3078 — SR T1 not found
[skip mask] sub-M2204_ses-577 — SR T1 not found
[skip mask] sub-M2208_ses-446 — SR T1 not found
[skip mask] sub-M2221_ses-2045 — SR T1 not found
[skip mask] sub-M2228_ses-456 — SR T1 not found
[skip mask] sub-M2246_ses-853 — SR T1 not found
[skip mask] sub-M2252_ses-738 — SR T1 not found
[skip mask] sub-M2281_ses-500 — SR T1 not found
[skip mask] su

In [ ]:
# Verification
print("=" * 50)
print("VERIFICATION")
print("=" * 50)

out_t1s   = list(OUT_T1_DIR.glob("*.nii.gz"))
out_masks = list(OUT_MASK_DIR.glob("*.nii.gz"))
print(f"Output T1s:   {len(out_t1s)}  (expected ~{len(src_t1s)})")
print(f"Output masks: {len(out_masks)}")

# Check a sample: nonzero %, shape
sample = out_t1s[0] if out_t1s else None
if sample:
    img = nib.load(str(sample))
    data = img.get_fdata(dtype=np.float32)
    nz_pct = 100 * np.count_nonzero(data) / data.size
    print(f"\nSample: {sample.name}")
    print(f"  Shape: {img.shape}")
    print(f"  Nonzero: {nz_pct:.1f}%")
    print(f"  Value range: [{data.min():.4f}, {data.max():.4f}]")

VERIFICATION
Output T1s:   0  (expected ~198)
Output masks: 0
